# Social Media Habit Analyzer — SCRDR Experiment

This notebook demonstrates the full workflow of **Single Classification Ripple Down Rules (SCRDR)** applied to social media habit classification.

## Agenda
1. Dataset Loading & Exploration
2. Initial SCRDR Rule Tree
3. Step-by-Step Rule Evaluation
4. Batch Evaluation & Accuracy
5. Error Analysis
6. Incremental Learning (Rule Update)
7. Before vs After Comparison
8. Rule Tree Growth

In [2]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
from scrdr_engine import load_rules_from_json, evaluate_scrdr, match_condition, trace_path
from evaluator import load_dataset, evaluate_dataset, print_prediction, print_summary
from rule_updater import find_last_fired_rule, add_exception_rule, save_rules_to_json
from utils import print_all_rules, confusion_matrix, print_confusion_matrix, per_class_accuracy, format_input
print('All modules loaded successfully.')

All modules loaded successfully.


## 1. Dataset Loading & Exploration

In [3]:
df = pd.read_csv('../data/social_media_dataset.csv')
print(f'Dataset shape: {df.shape}')
df.head(10)

Dataset shape: (45, 7)


,id,daily_usage_hours,sleep_time,primary_usage_type,notification_check_frequency,purpose,label
0,1,1.5,early,study,low,learning,Healthy
1,2,2.0,normal,study,low,learning,Healthy
2,3,1.0,early,social,low,communication,Healthy
3,4,2.5,normal,study,medium,learning,Healthy
4,5,1.5,early,study,low,communication,Healthy
5,6,2.0,normal,social,low,communication,Healthy
6,7,1.0,early,study,low,learning,Healthy
7,8,3.0,normal,study,medium,learning,Healthy
8,9,2.0,early,social,low,learning,Healthy
9,10,1.5,normal,study,low,communication,Healthy


In [4]:
print('Class Distribution:')
print(df['label'].value_counts())
print('\nAvg usage hours by class:')
print(df.groupby('label')['daily_usage_hours'].mean())

Class Distribution:
label
Healthy      15
Moderate     15
Unhealthy    15
Name: count, dtype: int64

Avg usage hours by class:
label
Healthy      1.826667
Moderate     3.680000
Unhealthy    6.053333
Name: daily_usage_hours, dtype: float64


## 2. Initial SCRDR Rule Tree

In [5]:
rules = load_rules_from_json('../rules/scrdr_rules.json')
print(f'Number of root rule chains: {len(rules)}')
for r in rules:
    print(f'  Root: [{r.id}] -> {r.conclusion} (conditions: {len(r.conditions)})')
    if r.exception:
        print(f'    Exception: [{r.exception.id}] -> {r.exception.conclusion}')

Number of root rule chains: 3
  Root: [R2] -> Unhealthy (conditions: 3)
    Exception: [R2a] -> Moderate
  Root: [R1] -> Healthy (conditions: 3)
    Exception: [R1a] -> Moderate
  Root: [R0] -> Moderate (conditions: 0)


In [6]:
print_all_rules(rules)


SCRDR RULE TREE
[R2] → Unhealthy
   • daily_usage_hours > 5.0
   • sleep_time == late
   • notification_check_frequency == high
   ✎ High usage (>5h), late-night sleep, and compulsive notification checking indicate disruptive social media behavior — classified as Unhealthy.
    ↳ [R2a] → Moderate
       • primary_usage_type == study
       • purpose == learning
       ✎ Exception: Heavy usage driven by academic study and learning reduces severity — reclassified as Moderate despite late sleep and high notifications.

[R1] → Healthy
   • daily_usage_hours <= 3.0
   • sleep_time in ['early', 'normal']
   • purpose in ['learning', 'communication']
   ✎ Low usage (<=3h/day), good sleep schedule, and purposeful usage (learning or communication) indicates a Healthy social media habit.
    ↳ [R1a] → Moderate
       • notification_check_frequency == high
       • primary_usage_type == social
       ✎ Exception: Even with low hours, high notification frequency combined with purely social usage 

## 3. Step-by-Step Rule Evaluation

Let's manually trace through several examples to understand SCRDR traversal.

In [7]:
# EXAMPLE 1: Healthy user
print('=== EXAMPLE 1: Healthy User ===')
healthy_user = format_input(1.5, 'early', 'study', 'low', 'learning')
print('Input:', healthy_user)
result = evaluate_scrdr(healthy_user, rules)
print(f'\nPrediction  : {result["label"]}')
print(f'Rule Path   : {" > ".join(result["rule_path"])}')
print(f'Explanation : {result["explanation"]}')

=== EXAMPLE 1: Healthy User ===
Input: {'daily_usage_hours': 1.5, 'sleep_time': 'early', 'primary_usage_type': 'study', 'notification_check_frequency': 'low', 'purpose': 'learning'}

Prediction  : Healthy
Rule Path   : R1
Explanation : Low usage (<=3h/day), good sleep schedule, and purposeful usage (learning or communication) indicates a Healthy social media habit.


In [8]:
# EXAMPLE 2: Unhealthy user
print('=== EXAMPLE 2: Unhealthy User ===')
unhealthy_user = format_input(7.0, 'late', 'entertainment', 'high', 'scrolling')
print('Input:', unhealthy_user)
result = evaluate_scrdr(unhealthy_user, rules)
print(f'\nPrediction  : {result["label"]}')
print(f'Rule Path   : {" > ".join(result["rule_path"])}')
print(f'Explanation : {result["explanation"]}')

=== EXAMPLE 2: Unhealthy User ===
Input: {'daily_usage_hours': 7.0, 'sleep_time': 'late', 'primary_usage_type': 'entertainment', 'notification_check_frequency': 'high', 'purpose': 'scrolling'}

Prediction  : Unhealthy
Rule Path   : R2
Explanation : High usage (>5h), late-night sleep, and compulsive notification checking indicate disruptive social media behavior — classified as Unhealthy.


In [9]:
# EXAMPLE 3: Exception case - Heavy usage but for study (R2a fires)
print('=== EXAMPLE 3: Heavy Studying - Exception R2a fires ===')
heavy_studier = format_input(6.0, 'late', 'study', 'high', 'learning')
print('Input:', heavy_studier)
result = evaluate_scrdr(heavy_studier, rules)
print(f'\nPrediction  : {result["label"]}  <- Softened by R2a exception!')
print(f'Rule Path   : {" > ".join(result["rule_path"])}')
print(f'Explanation : {result["explanation"]}')

=== EXAMPLE 3: Heavy Studying - Exception R2a fires ===
Input: {'daily_usage_hours': 6.0, 'sleep_time': 'late', 'primary_usage_type': 'study', 'notification_check_frequency': 'high', 'purpose': 'learning'}

Prediction  : Moderate  <- Softened by R2a exception!
Rule Path   : R2 > R2a
Explanation : Exception: Heavy usage driven by academic study and learning reduces severity — reclassified as Moderate despite late sleep and high notifications.


In [10]:
# EXAMPLE 4: Low hours but high notifications (R1a fires)
print('=== EXAMPLE 4: Low Hours but High Notifications - R1a fires ===')
notif_addict = format_input(2.5, 'normal', 'social', 'high', 'communication')
print('Input:', notif_addict)
result = evaluate_scrdr(notif_addict, rules)
print(f'\nPrediction  : {result["label"]}  <- Raised by R1a exception!')
print(f'Rule Path   : {" > ".join(result["rule_path"])}')
print(f'Explanation : {result["explanation"]}')

=== EXAMPLE 4: Low Hours but High Notifications - R1a fires ===
Input: {'daily_usage_hours': 2.5, 'sleep_time': 'normal', 'primary_usage_type': 'social', 'notification_check_frequency': 'high', 'purpose': 'communication'}

Prediction  : Moderate  <- Raised by R1a exception!
Rule Path   : R1 > R1a
Explanation : Exception: Even with low hours, high notification frequency combined with purely social usage indicates dependency signals — reclassified as Moderate.


## 4. Batch Evaluation & Accuracy

In [11]:
cases = load_dataset('../data/social_media_dataset.csv')
eval_result = evaluate_dataset(cases, rules)
print_summary(eval_result)

SCRDR EVALUATION SUMMARY
Total Samples : 45
Correct       : 42
Accuracy      : 93.3%
Errors        : 3


In [12]:
cm = confusion_matrix(eval_result['results'])
print_confusion_matrix(cm)

pa = per_class_accuracy(cm)
print('Per-class accuracy:')
for cls, acc in pa.items():
    bar = 'X' * int(acc * 20)
    print(f'  {cls:12s}: {acc*100:5.1f}% [{bar}]')


Confusion Matrix (True \ Predicted):
                 Healthy    Moderate   Unhealthy
------------------------------------------------
     Healthy          15           0           0
    Moderate           0          15           0
   Unhealthy           0           3          12

Per-class accuracy:
  Healthy     : 100.0% [XXXXXXXXXXXXXXXXXXXX]
  Moderate    : 100.0% [XXXXXXXXXXXXXXXXXXXX]
  Unhealthy   :  80.0% [XXXXXXXXXXXXXXXX]


In [13]:
# Show all predictions as a DataFrame
results_df = pd.DataFrame([
    {'id': r['id'], 'true': r['true_label'], 'predicted': r['predicted'],
     'correct': 'YES' if r['correct'] else 'NO',
     'rule_path': ' -> '.join(r['rule_path'])}
    for r in eval_result['results']
])
results_df

,id,true,predicted,correct,rule_path
0,1,Healthy,Healthy,YES,R1
1,2,Healthy,Healthy,YES,R1
2,3,Healthy,Healthy,YES,R1
3,4,Healthy,Healthy,YES,R1
4,5,Healthy,Healthy,YES,R1
5,6,Healthy,Healthy,YES,R1
6,7,Healthy,Healthy,YES,R1
7,8,Healthy,Healthy,YES,R1
8,9,Healthy,Healthy,YES,R1
9,10,Healthy,Healthy,YES,R1


## 5. Error Analysis

In [14]:
print(f'Total errors: {len(eval_result["errors"])}')
for e in eval_result['errors']:
    print(f'\nID {e["id"]}:')
    print(f'  Input     : {e["input"]}')
    print(f'  True      : {e["true_label"]}')
    print(f'  Predicted : {e["predicted"]}')
    print(f'  Rule Path : {" -> ".join(e["rule_path"])}')
    print('  Root Cause: daily_usage_hours is at boundary (4.8-5.0); R2 requires > 5.0 (strict)')

Total errors: 3

ID 21:
  Input     : {'id': '21', 'daily_usage_hours': 5.0, 'sleep_time': 'late', 'primary_usage_type': 'entertainment', 'notification_check_frequency': 'high', 'purpose': 'scrolling'}
  True      : Unhealthy
  Predicted : Moderate
  Rule Path : R0
  Root Cause: daily_usage_hours is at boundary (4.8-5.0); R2 requires > 5.0 (strict)

ID 26:
  Input     : {'id': '26', 'daily_usage_hours': 5.0, 'sleep_time': 'late', 'primary_usage_type': 'entertainment', 'notification_check_frequency': 'high', 'purpose': 'scrolling'}
  True      : Unhealthy
  Predicted : Moderate
  Rule Path : R0
  Root Cause: daily_usage_hours is at boundary (4.8-5.0); R2 requires > 5.0 (strict)

ID 39:
  Input     : {'id': '39', 'daily_usage_hours': 4.8, 'sleep_time': 'late', 'primary_usage_type': 'entertainment', 'notification_check_frequency': 'high', 'purpose': 'scrolling'}
  True      : Unhealthy
  Predicted : Moderate
  Rule Path : R0
  Root Cause: daily_usage_hours is at boundary (4.8-5.0); R2 req

## 6. Incremental Learning — Adding Exception Rules

SCRDR's key innovation: correct mistakes by **adding** exceptions, never modifying existing rules.

In [15]:
# Reload fresh rules
rules_updated = load_rules_from_json('../rules/scrdr_rules.json')

misclassified_case = {
    'daily_usage_hours': 5.0, 'sleep_time': 'late',
    'primary_usage_type': 'entertainment',
    'notification_check_frequency': 'high', 'purpose': 'scrolling'
}

print('BEFORE FIX:')
result_before = evaluate_scrdr(misclassified_case, rules_updated)
print(f'  Prediction: {result_before["label"]}  (should be Unhealthy)')

BEFORE FIX:
  Prediction: Moderate  (should be Unhealthy)


In [16]:
# Find last fired rule and add exception
last_fired = find_last_fired_rule(misclassified_case, rules_updated)
print(f'Last fired rule: [{last_fired.id}] -> {last_fired.conclusion}')

new_rule = add_exception_rule(
    last_fired=last_fired,
    new_conditions={
        'daily_usage_hours': {'op': '>=', 'value': 4.5},
        'sleep_time': {'op': '==', 'value': 'late'},
        'notification_check_frequency': {'op': '==', 'value': 'high'},
        'primary_usage_type': {'op': '==', 'value': 'entertainment'}
    },
    new_conclusion='Unhealthy',
    new_justification='Exception: 4.5+ hours of entertainment with late sleep and high notifications is Unhealthy even below 5h threshold.',
    new_rule_id='R0_boundary_unhealthy'
)
print(f'Added rule: [{new_rule.id}]')

Last fired rule: [R0] -> Moderate
[RuleUpdater] Added exception rule 'R0_boundary_unhealthy' under 'R0'
             Conclusion: Unhealthy
             Conditions: {'daily_usage_hours': {'op': '>=', 'value': 4.5}, 'sleep_time': {'op': '==', 'value': 'late'}, 'notification_check_frequency': {'op': '==', 'value': 'high'}, 'primary_usage_type': {'op': '==', 'value': 'entertainment'}}
Added rule: [R0_boundary_unhealthy]


In [17]:
print('AFTER FIX:')
result_after = evaluate_scrdr(misclassified_case, rules_updated)
print(f'  Prediction  : {result_after["label"]}')
print(f'  Rule Path   : {" -> ".join(result_after["rule_path"])}')
print(f'  Explanation : {result_after["explanation"]}')

AFTER FIX:
  Prediction  : Unhealthy
  Rule Path   : R0 -> R0_boundary_unhealthy
  Explanation : Exception: 4.5+ hours of entertainment with late sleep and high notifications is Unhealthy even below 5h threshold.


## 7. Before vs After Comparison

In [18]:
eval_after = evaluate_dataset(cases, rules_updated)

print('BEFORE adding exception rule:')
print(f'  Accuracy: {eval_result["accuracy"]*100:.1f}%  ({eval_result["correct"]}/{eval_result["total"]})')
print(f'  Errors  : {len(eval_result["errors"])}')
print()
print('AFTER adding exception rule:')
print(f'  Accuracy: {eval_after["accuracy"]*100:.1f}%  ({eval_after["correct"]}/{eval_after["total"]})')
print(f'  Errors  : {len(eval_after["errors"])}')
print(f'\nImprovement: +{(eval_after["accuracy"] - eval_result["accuracy"])*100:.1f}%')

BEFORE adding exception rule:
  Accuracy: 93.3%  (42/45)
  Errors  : 3

AFTER adding exception rule:
  Accuracy: 100.0%  (45/45)
  Errors  : 0

Improvement: +6.7%


## 8. Rule Tree Growth

In [19]:
def count_rules(rule_list):
    total = 0
    def traverse(rule):
        nonlocal total
        if rule is None: return
        total += 1
        traverse(rule.exception)
    for r in rule_list:
        traverse(r)
    return total

rules_initial = load_rules_from_json('../rules/scrdr_rules.json')
n_before = count_rules(rules_initial)
n_after = count_rules(rules_updated)

print(f'Initial rule count : {n_before}')
print(f'Final rule count   : {n_after}')
print(f'Rules added        : {n_after - n_before} (existing rules UNCHANGED)')

Initial rule count : 5
Final rule count   : 6
Rules added        : 1 (existing rules UNCHANGED)


In [20]:
print('UPDATED RULE TREE:')
print_all_rules(rules_updated)

UPDATED RULE TREE:

SCRDR RULE TREE
[R2] → Unhealthy
   • daily_usage_hours > 5.0
   • sleep_time == late
   • notification_check_frequency == high
   ✎ High usage (>5h), late-night sleep, and compulsive notification checking indicate disruptive social media behavior — classified as Unhealthy.
    ↳ [R2a] → Moderate
       • primary_usage_type == study
       • purpose == learning
       ✎ Exception: Heavy usage driven by academic study and learning reduces severity — reclassified as Moderate despite late sleep and high notifications.

[R1] → Healthy
   • daily_usage_hours <= 3.0
   • sleep_time in ['early', 'normal']
   • purpose in ['learning', 'communication']
   ✎ Low usage (<=3h/day), good sleep schedule, and purposeful usage (learning or communication) indicates a Healthy social media habit.
    ↳ [R1a] → Moderate
       • notification_check_frequency == high
       • primary_usage_type == social
       ✎ Exception: Even with low hours, high notification frequency combined with p

In [21]:
print('='*60)
print('SCRDR EXPERIMENT COMPLETE')
print('='*60)
print(f'Dataset: 45 samples | 3 balanced classes')
print(f'Initial accuracy : {eval_result["accuracy"]*100:.1f}%')
print(f'Final accuracy   : {eval_after["accuracy"]*100:.1f}%')
print(f'Rules grown from : {n_before} -> {n_after}')
print('Every prediction: Fully explainable with rule path + justification')

SCRDR EXPERIMENT COMPLETE
Dataset: 45 samples | 3 balanced classes
Initial accuracy : 93.3%
Final accuracy   : 100.0%
Rules grown from : 5 -> 6
Every prediction: Fully explainable with rule path + justification
